In [16]:
import os
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
# Load paths from .env file
data_dir = os.getenv("DATA_DIR")
embeddings_dir = os.getenv("EMBEDDINGS_DIR")

# Load the unified TSV file with image hashes
annotations_file = os.path.join(data_dir, "preprocessed_annotations.tsv")

In [19]:
df = pd.read_csv(annotations_file, sep="\t")

In [22]:
df.head()

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,...,general_disaster_type,img_width,img_height,img_aspect_ratio,img_hash_str,force_test,mm_info,mm_human,cleaned_text_bert,cleaned_text_bertweet
0,917791044158185473,917791044158185473_0,informative,1.0000,informative,0.6766,other_relevant_information,1.0000,other_relevant_information,0.6766,...,wildfire,800,450,1.777778,86c476bb39c2b16c,False,informative,other_relevant_information,Wildfires raging through Northern California a...,Wildfires raging through Northern California a...
1,917791130590183424,917791130590183424_0,informative,1.0000,informative,0.6667,infrastructure_and_utility_damage,1.0000,affected_individuals,0.6667,...,wildfire,1200,677,1.772526,f28e9aa98b96a730,True,informative,conflict,PHOTOS: Deadly wildfires rage in California,PHOTOS: Deadly wildfires rage in California ht...
2,917791291823591425,917791291823591425_0,informative,0.6813,informative,1.0000,other_relevant_information,0.6813,infrastructure_and_utility_damage,1.0000,...,wildfire,640,480,1.333333,a9bdb18391838bba,True,informative,infrastructure_and_utility_damage,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
3,917791291823591425,917791291823591425_1,informative,0.6813,not_informative,1.0000,other_relevant_information,0.6813,not_humanitarian,1.0000,...,wildfire,1200,900,1.333333,b582566fab45cac8,True,informative,other_relevant_information,"PLS SHARE: We're capturing wildfire response, ...","PLS SHARE: We're capturing wildfire response, ..."
4,917792092100988929,917792092100988929_0,informative,0.6727,informative,0.6612,other_relevant_information,0.6727,infrastructure_and_utility_damage,0.6612,...,wildfire,600,400,1.500000,86826be7394ed43a,False,informative,infrastructure_and_utility_damage,California's raging wildfires as you've never ...,California's raging wildfires as you've never ...


In [20]:
def generate_bert_embeddings(df, text_column, model_name="bert-base-uncased", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="BERT embeddings"):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()  # [CLS] token
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


def generate_roberta_embeddings(df, text_column, model_name="roberta-base", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="RoBERTa embeddings"):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


def generate_bertweet_embeddings(df, text_column, model_name="vinai/bertweet-base", batch_size=16, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModel.from_pretrained(model_name)
    model.eval()

    embeddings = []

    for i in tqdm(range(0, len(df), batch_size), desc="BERTweet embeddings"):
        batch_texts = df[text_column].iloc[i:i+batch_size].tolist()
        encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embeddings)

    return np.vstack(embeddings)


In [23]:
os.makedirs(embeddings_dir, exist_ok=True)

bert_vecs = generate_bert_embeddings(df, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "embeddings_bert.npy"), bert_vecs)

roberta_vecs = generate_roberta_embeddings(df, "cleaned_text_bert")
np.save(os.path.join(embeddings_dir, "embeddings_roberta.npy"), roberta_vecs)

bertweet_vecs = generate_bertweet_embeddings(df, "cleaned_text_bertweet")
np.save(os.path.join(embeddings_dir, "embeddings_bertweet.npy"), bertweet_vecs)

BERT embeddings:   2%|▏         | 18/1131 [00:15<15:46,  1.18it/s]


KeyboardInterrupt: 